# CETR-Net V2 migrated full-data refinement — 42 / routing
Kaggle account agusperdana. Preserves provenance of independently trained frozen encoder on original ardhanabdillah account. FIVES V2 split 508 train/90 val. This is validation-only and uses no official test data.
Predeclared variant group: routing, 8 epochs/variant. All training on Kaggle GPU, not laptop GPU.


In [ ]:
import os,sys,json,csv,zipfile,shutil,time
from pathlib import Path
os.environ["TF_CPP_MIN_LOG_LEVEL"]="2"
import numpy as np,cv2,tensorflow as tf
import skimage
gpus=tf.config.list_physical_devices("GPU")
print("STAGE2_ENV",{"tf":tf.__version__,"gpu":str(gpus),"skimage":skimage.__version__},flush=True)
assert gpus,"GPU unavailable"
for gpu in gpus:
    try:tf.config.experimental.set_memory_growth(gpu,True)
    except RuntimeError: pass
INPUT=Path("/kaggle/input/datasets/agusperdana/cetr-fives-v2-private-dev")
if not INPUT.exists():
    for candidate in (Path("/kaggle/input/cetr-fives-v2-private-dev"),):
        if candidate.exists(): INPUT=candidate
if not INPUT.exists():
    matches=[]
    for here,ds,fs in os.walk("/kaggle/input",followlinks=True):
        if len(Path(here).relative_to("/kaggle/input").parts)>7:
            ds.clear()
            continue
        if "fives_v2_trainval_512.zip" in fs:matches.append(Path(here))
    print("FOUND_INPUTS",[str(m) for m in matches])
    if len(matches)==1:INPUT=matches[0]
assert INPUT.exists(),"Private dataset not mounted"
WORK=Path("/kaggle/working/cetr_v2")
WORK.mkdir(parents=True,exist_ok=True)
print("DATASET_PATH",INPUT,flush=True)


In [ ]:
raw=INPUT/"fives_v2_trainval_512"
if raw.is_dir() and (raw/"record_manifest.json").exists():
    print("KAGGLE_DATASET_EXTRACTED_FOLDER",raw,flush=True)
else:
    archives=list(INPUT.rglob("fives_v2_trainval_512.zip"))
    assert len(archives)==1,"No FIVES V2 image package found"
    raw=WORK/"raw"
    raw.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(raw)
manifest=json.loads((raw/"record_manifest.json").read_text())
splits=json.loads((raw/"seed_splits.json").read_text())
mapping={r["id"]:r for r in manifest}
assert len(mapping)==598
for seed in ("42","123","2025"):
    train=splits[seed]["train"]
    val=splits[seed]["validation"]
    assert len(train)==508 and len(val)==90 and not set(train).intersection(val)
    folder=WORK/"SPLITS_NEARDUP_SAFE_V2"/("seed_"+seed)
    folder.mkdir(parents=True,exist_ok=True)
    for split_name,ids in (("train",train),("validation",val)):
        with (folder/("fives_"+split_name+"_split.csv")).open("w",newline="",encoding="utf8") as f:
            wr=csv.DictWriter(f,fieldnames=["id","image","mask","official_split","label_token"])
            wr.writeheader()
            for rid in ids:
                row=mapping[rid]
                img=raw/"images"/(row["image_key"]+".png")
                mask=raw/"masks"/(rid+".png")
                assert img.is_file() and mask.is_file()
                wr.writerow({"id":rid,"image":str(img),"mask":str(mask),
                             "official_split":"official_train","label_token":row["label"]})
print("FIVES_V2_SPLITS_OK",flush=True)


In [ ]:
from pathlib import Path
WORK=Path('/kaggle/working/cetr_v2')
(WORK/'cetr_core_ablation.py').write_text('"""CETR-Net experimental prototype, NOT a completed peer-reviewed result.\nReal fundus RGB is required at inference. Annotation-derived perturbations are TRAINING-ONLY.\nNo operations write into historical EAG-UNet folders.\n"""\nimport math\nimport cv2\nimport numpy as np\nimport tensorflow as tf\nfrom skimage.morphology import skeletonize\n\n\ndef conv_block(x, ch):\n    for _ in range(2):\n        x = tf.keras.layers.Conv2D(ch, 3, padding="same", use_bias=False)(x)\n        x = tf.keras.layers.BatchNormalization()(x)\n        x = tf.keras.layers.Activation("relu")(x)\n    return x\n\n\ndef make_backbone(shape=(256, 256, 3)):\n    image = tf.keras.Input(shape=shape, name="RGB_fundus")\n    x, skips = image, []\n    for c in (24, 48, 96, 192):\n        x = conv_block(x, c)\n        skips.append(x)\n        x = tf.keras.layers.MaxPool2D()(x)\n    x = conv_block(x, 384)\n    for c, skip in zip((192, 96, 48, 24), reversed(skips)):\n        x = tf.keras.layers.UpSampling2D(interpolation="bilinear")(x)\n        x = tf.keras.layers.Concatenate()([x, skip])\n        x = conv_block(x, c)\n    p = tf.keras.layers.Conv2D(1, 1, activation="sigmoid", name="P0")(x)\n    return tf.keras.Model(image, [p, x], name="cetr_initial_encoder_decoder")\n\n\ndef line_points(a, b, count=16):\n    yy = np.rint(np.linspace(a[0], b[0], count)).astype(int)\n    xx = np.rint(np.linspace(a[1], b[1], count)).astype(int)\n    return np.stack([yy, xx], axis=-1)\n\n\ndef line_coverage(binary, a, b):\n    pts = line_points(a, b, 25)\n    return float(np.mean(binary[pts[:, 0], pts[:, 1]]))\n\n\ndef choose_pair(coords, Y, kind, rng, min_d=9, max_d=27):\n    if len(coords) == 0:\n        raise ValueError("No skeleton coordinates; cannot construct topology examples")\n    for _ in range(1800):\n        a = coords[int(rng.integers(len(coords)))]\n        b = coords[int(rng.integers(len(coords)))]\n        dist = float(np.linalg.norm(a-b))\n        if dist < min_d or dist > max_d: continue\n        ratio = line_coverage(Y, a, b)\n        if kind == "true" and ratio >= .78: return tuple(a), tuple(b)\n        if kind == "false" and ratio <= .30: return tuple(a), tuple(b)\n    raise ValueError("No eligible "+kind+" candidate; skip rather than fabricate training labels")\n\n\ndef candidate(a, b, kind, label, shape):\n    h, w = shape\n    path = np.clip(line_points(a, b, 20), [0, 0], [h-1, w-1]).astype("int32")\n    ker = np.zeros((h, w), "float32")\n    cv2.line(ker, (int(a[1]), int(a[0])), (int(b[1]), int(b[0])), 1, 3, cv2.LINE_8)\n    ker = cv2.GaussianBlur(ker, (5,5), .7)\n    ker = np.clip(ker, 0, 1)\n    return {"path":path, "kernel":ker, "kind":kind, "label":float(label), "a":a, "b":b}\n\n\ndef supervised_candidates(y, rng):\n    """Training ONLY: construct four controls with ground-truth labels.\n    Missing true arc and added background bridge are supervised distinctly.\n    """\n    y = np.asarray(y>0.5, dtype=np.uint8)\n    h, w = y.shape\n    sk = skeletonize(y>0).astype("uint8")\n    coords = np.argwhere(sk>0)\n    ar,br = choose_pair(coords,y,"true",rng)\n    af,bf = choose_pair(coords,y,"false",rng)\n    # Other two are negative action controls: nothing is corrupted along them.\n    candidates = [\n        candidate(ar,br,"restore",1,(h,w)),\n        candidate(af,bf,"reject",1,(h,w)),\n        candidate(af,bf,"restore",0,(h,w)),\n        candidate(ar,br,"reject",0,(h,w)),\n    ]\n    removed = ((candidates[0]["kernel"]>.5)&(y==1)).astype("uint8")\n    inserted = ((candidates[1]["kernel"]>.5)&(y==0)).astype("uint8")\n    if (removed & (1-y)).any() or (inserted & y).any():\n        raise AssertionError("counterfactual masks violate Y constraints")\n    corrupted = (y * (1-removed)) | inserted\n    return corrupted.astype("float32"), candidates, removed, inserted\n\n\ndef inference_candidates(preliminary_binary, max_candidates=12, radius=25):\n    """Inference depends only on P0, no ground-truth or other labels."""\n    mask = np.asarray(preliminary_binary>0.5, dtype="uint8")\n    h,w = mask.shape\n    sk = skeletonize(mask>0).astype("uint8")\n    n = cv2.filter2D(sk,-1,np.ones((3,3),np.uint8),borderType=cv2.BORDER_CONSTANT)-sk\n    endpoints = np.argwhere((sk>0)&(n==1))\n    pts = np.argwhere(sk>0)\n    proposals=[]; seen=set()\n    for i,a in enumerate(endpoints):\n        for b in endpoints[i+1:]:\n            dis=np.linalg.norm(a-b)\n            if 7<dis<radius:\n                k=tuple(sorted((tuple(a),tuple(b))))\n                if k not in seen:\n                    proposals.append(candidate(tuple(a),tuple(b),"restore",0,(h,w)))\n                    seen.add(k)\n            if len(proposals)>=max_candidates//2:break\n        if len(proposals)>=max_candidates//2:break\n    # Existing centerline segments are possible false-link candidates.\n    for i in np.linspace(0,max(0,len(pts)-1),min(len(pts),max_candidates*2),dtype=int):\n        if len(proposals)>=max_candidates:break\n        a=pts[i]\n        near=pts[np.sum((pts-a)**2,axis=1)<400]\n        if len(near)<2: continue\n        b=near[-1]\n        if 7<np.linalg.norm(a-b)<radius:\n            proposals.append(candidate(tuple(a),tuple(b),"reject",0,(h,w)))\n    return proposals[:max_candidates]\n\n\nclass CETRNet(tf.keras.Model):\n    def __init__(self, shape=(256,256,3)):\n        super().__init__(name="CETRNet")\n        self.backbone=make_backbone(shape)\n        self.shared=tf.keras.layers.Dense(64,activation="relu")\n        self.restore=tf.keras.layers.Dense(1,activation="sigmoid",name="restore_head")\n        self.reject=tf.keras.layers.Dense(1,activation="sigmoid",name="reject_head")\n\n    def forward(self, image, candidate_list, proposal_mask, training=False, evidence_mode="full"):\n        """RGB image [1,H,W,3]; proposal_mask [H,W], graph from predictions or training perturbations."""\n        p0, feat=self.backbone(image,training=training)\n        H,W=int(p0.shape[1]),int(p0.shape[2])\n        if not candidate_list:\n            return p0,p0,tf.zeros((0,),tf.float32),tf.zeros((0,),tf.float32),tf.constant(0.)\n        image0=image[0]; p00=p0[0]; f0=feat[0]\n        desc=[]\n        for e in candidate_list:\n            idx=tf.constant(e["path"],dtype=tf.int32)\n            rgb=tf.reduce_mean(tf.gather_nd(image0,idx),axis=0)\n            featvec=tf.reduce_mean(tf.gather_nd(f0,idx),axis=0)\n            prob=tf.gather_nd(p00,idx)[:,0]\n            ent=-prob*tf.math.log(prob+1e-6)-(1-prob)*tf.math.log(1-prob+1e-6)\n            dy=(e["b"][0]-e["a"][0])/H\n            dx=(e["b"][1]-e["a"][1])/W\n            geom=tf.constant([dy,dx,math.sqrt(dy*dy+dx*dx)],tf.float32)\n            stats=tf.stack([tf.reduce_mean(prob),tf.math.reduce_std(prob),tf.reduce_mean(ent)])\n            if evidence_mode == "geometry":\n                rgb=tf.zeros_like(rgb);featvec=tf.zeros_like(featvec);stats=tf.zeros_like(stats)\n            elif evidence_mode == "confidence":\n                rgb=tf.zeros_like(rgb);featvec=tf.zeros_like(featvec)\n            elif evidence_mode == "image":\n                stats=tf.zeros_like(stats);geom=tf.zeros_like(geom)\n            elif evidence_mode != "full":\n                raise ValueError("Unrecognized evidence_mode: "+str(evidence_mode))\n            desc.append(tf.concat([rgb,featvec,stats,geom],axis=0))\n        z=self.shared(tf.stack(desc))\n        qr=tf.reshape(self.restore(z),(-1,))\n        qd=tf.reshape(self.reject(z),(-1,))\n        m=tf.constant(proposal_mask,dtype=tf.float32)\n        ker=tf.constant(np.stack([e["kernel"] for e in candidate_list]),tf.float32)\n        rt=tf.constant(np.array([float(e["kind"]=="restore") for e in candidate_list]),tf.float32)\n        dt=1.-rt\n        add=ker*(1.-m)[None,:,:]*rt[:,None,None]*qr[:,None,None]\n        delete=ker*m[None,:,:]*dt[:,None,None]*qd[:,None,None]\n        R=1.-tf.reduce_prod(1.-add,axis=0)\n        D=1.-tf.reduce_prod(1.-delete,axis=0)\n        if bool(tf.reduce_any(R*D>1e-6)):raise AssertionError("opposing pixel edits")\n        P=tf.clip_by_value(p00[:,:,0]+(1.-p00[:,:,0])*R-p00[:,:,0]*D,1e-5,1-1e-5)\n        return tf.expand_dims(P[None,:,:],-1),p0,qr,qd,tf.reduce_mean(R+D)\n\n\ndef soft_dice(y,p):\n    return (2*tf.reduce_sum(y*p)+1.)/(tf.reduce_sum(y)+tf.reduce_sum(p)+1.)\n\n\ndef _soft_erode(x):\n    return tf.minimum(-tf.nn.max_pool2d(-x,ksize=(3,1),strides=1,padding="SAME"),\n                      -tf.nn.max_pool2d(-x,ksize=(1,3),strides=1,padding="SAME"))\n\n\ndef _soft_open(x):\n    return tf.nn.max_pool2d(_soft_erode(x),ksize=3,strides=1,padding="SAME")\n\n\ndef _soft_skeleton(x,iters=8):\n    sk=tf.nn.relu(x-_soft_open(x))\n    for _ in range(iters):\n        x=_soft_erode(x)\n        delta=tf.nn.relu(x-_soft_open(x))\n        sk=sk+tf.nn.relu(delta-sk*delta)\n    return sk\n\n\ndef soft_cldice(y,p):\n    """Established soft-clDice formulation, NOT a new CETR-Net loss."""\n    sy=_soft_skeleton(y)\n    sp=_soft_skeleton(p)\n    prec=(tf.reduce_sum(sp*y)+1e-5)/(tf.reduce_sum(sp)+1e-5)\n    sens=(tf.reduce_sum(sy*p)+1e-5)/(tf.reduce_sum(sy)+1e-5)\n    return (2*prec*sens+1e-5)/(prec+sens+1e-5)\n\n\ndef training_loss(y,p,qr,qd,cands,top_weight=0.05,restore_weight=0.3,reject_weight=0.3):\n    seg=tf.reduce_mean(tf.keras.losses.binary_crossentropy(y,p))+(1-soft_dice(y,p))\n    restore_idx=[i for i,e in enumerate(cands) if e["kind"]=="restore"]\n    reject_idx=[i for i,e in enumerate(cands) if e["kind"]=="reject"]\n    def action_loss(scores,indices):\n        if not indices:return tf.constant(0.,tf.float32)\n        pred=tf.gather(scores,indices)\n        truth=tf.constant([cands[i]["label"] for i in indices],tf.float32)\n        return tf.reduce_mean(tf.keras.losses.binary_crossentropy(truth,pred))\n    lr=action_loss(qr,restore_idx)\n    ld=action_loss(qd,reject_idx)\n    lt=1-soft_cldice(y,p) if top_weight>0 else tf.constant(0.,tf.float32)\n    return seg+restore_weight*lr+reject_weight*ld+top_weight*lt,{"seg":seg,"restore":lr,"reject":ld,"topology":lt}\n',encoding='utf8')
(WORK/'cetr_v2_suite.py').write_text('"""CETR-Net V2 topology-stage experimental runner.\nTrain/validation ONLY; accepts prepared /kaggle/working/cetr_v2 or local WSL WORK.\nPilot phase is NOT a final paper result; complete multi-seed independently initialized backbones and official test are a separate locked evaluation.\n"""\nimport os, json, csv, time, random, hashlib, argparse, traceback\nfrom pathlib import Path\nimport numpy as np, cv2, tensorflow as tf\nfrom skimage.morphology import skeletonize\nfrom cetr_core_ablation import (\n    CETRNet, supervised_candidates, inference_candidates, training_loss, line_coverage, soft_dice\n)\nVARIANTS={\n "natural_full_symmetric":dict(cdes="natural",evidence="full",wr=.3,wd=.3),\n "break_only":dict(cdes="break",evidence="full",wr=.3,wd=.3),\n "false_only":dict(cdes="false",evidence="full",wr=.3,wd=.3),\n "dual_geometry":dict(cdes="dual",evidence="geometry",wr=.3,wd=.3),\n "dual_confidence":dict(cdes="dual",evidence="confidence",wr=.3,wd=.3),\n "dual_full_symmetric":dict(cdes="dual",evidence="full",wr=.3,wd=.3),\n "dual_full_asymmetric":dict(cdes="dual",evidence="full",wr=.45,wd=.75),\n}\ndef get_rows(work,seed,part):\n    p=work/"SPLITS_NEARDUP_SAFE_V2"/("seed_"+str(seed))/("fives_"+part+"_split.csv")\n    with p.open(encoding="utf8",newline="") as f:return list(csv.DictReader(f))\ndef load(row,size=512):\n    img=cv2.imread(row["image"],cv2.IMREAD_COLOR)\n    mask=cv2.imread(row["mask"],cv2.IMREAD_GRAYSCALE)\n    if img is None or mask is None:raise FileNotFoundError((row["image"],row["mask"]))\n    img=cv2.cvtColor(cv2.resize(img,(size,size)),cv2.COLOR_BGR2RGB).astype("float32")/255.\n    mask=(cv2.resize(mask,(size,size),interpolation=cv2.INTER_NEAREST)>127).astype("float32")\n    return tf.convert_to_tensor(img[None]),tf.convert_to_tensor(mask[None,:,:,None])\ndef candidates_from_model(pred,truth):\n    mask=(pred>=.5).astype("float32")\n    cands=inference_candidates(mask,max_candidates=12)\n    scored=[]\n    for c in cands:\n        gt=line_coverage(truth,c["a"],c["b"])\n        if .35<gt<.7:continue\n        c["label"]=float(gt>=.7) if c["kind"]=="restore" else float(gt<=.35)\n        scored.append(c)\n    return mask,scored\ndef candidates_for_inference(pred):\n    """Inference proposals MUST be independent from ground-truth annotations.\n    Only preliminary prediction may determine retained candidate paths.\n    """\n    mask=(pred>=.5).astype("float32")\n    cands=inference_candidates(mask,max_candidates=12)\n    return mask,cands\n\ndef candidates_from_cdes(pred,truth,rng,mode):\n    mask=(pred>=.5).astype("float32")\n    try:\n        _,original,removed,inserted=supervised_candidates(truth,rng)\n    except ValueError:\n        return candidates_from_model(pred,truth)\n    original=[c for c in original if \n         (mode=="dual" or (mode=="break" and c["kind"]=="restore") or \n          (mode=="false" and c["kind"]=="reject"))]\n    if mode in ("break","dual"):mask[removed>0]=0.\n    if mode in ("false","dual"):mask[inserted>0]=1.\n    return mask,original\ndef scores(y,p):\n    pred=np.asarray(p>.5)\n    truth=np.asarray(y>.5)\n    inter=np.logical_and(pred,truth).sum()\n    dice=float((2*inter+1)/(pred.sum()+truth.sum()+1))\n    iou=float((inter+1)/(np.logical_or(pred,truth).sum()+1))\n    skpred=skeletonize(pred)\n    sktruth=skeletonize(truth)\n    a=float((skpred&truth).sum()+1e-5)/(float(skpred.sum())+1e-5)\n    b=float((sktruth&pred).sum()+1e-5)/(float(sktruth.sum())+1e-5)\n    cl=float((2*a*b+1e-5)/(a+b+1e-5))\n    return {"dice":dice,"iou":iou,"cldice":cl}\ndef atomic(path,obj):\n    temp=Path(str(path)+".tmp")\n    temp.write_text(json.dumps(obj,indent=2),encoding="utf8")\n    temp.replace(path)\ndef run_one(work,weight_path,seed,variant,epochs,train_n,val_n,size,report_root):\n    cfg=VARIANTS[variant]\n    train=get_rows(work,seed,"train")\n    val=get_rows(work,seed,"validation")\n    assert len(train)==508 and len(val)==90 and size==512\n    if train_n>0:train=train[:min(train_n,len(train))]\n    if val_n>0:val=val[:min(val_n,len(val))]\n    tf.keras.backend.clear_session()\n    tf.random.set_seed(seed)\n    random.seed(seed);rng=np.random.default_rng(seed)\n    net=CETRNet((size,size,3))\n    net.backbone.load_weights(str(weight_path))\n    net.backbone.trainable=False\n    optim=tf.keras.optimizers.Adam(3e-4)\n    out=report_root/("seed_"+str(seed))/variant\n    out.mkdir(parents=True,exist_ok=True)\n    history=[]\n    now=time.time()\n    init=load(train[0],size)\n    p0,_=net.backbone(init[0],training=False)\n    mask,cands=candidates_from_cdes(p0.numpy()[0,:,:,0],init[1].numpy()[0,:,:,0],rng,"dual")\n    net.forward(init[0],cands,mask,training=False,evidence_mode=cfg["evidence"])\n    epvar=tf.Variable(0,dtype=tf.int64,trainable=False)\n    ckpt=tf.train.Checkpoint(shared=net.shared,restore_head=net.restore,\n                              reject_head=net.reject,optimizer=optim,epoch=epvar)\n    manager=tf.train.CheckpointManager(ckpt,str(out/"resume_checkpoint"),max_to_keep=2)\n    if manager.latest_checkpoint:\n        ckpt.restore(manager.latest_checkpoint).expect_partial()\n        print("REFINER_RESUME",seed,variant,manager.latest_checkpoint,flush=True)\n    best_path=out/"best_state.json"\n    best=json.loads(best_path.read_text(encoding="utf8")) if best_path.exists() else {"best_dice":-1.,"best_epoch":0}\n    for epoch in range(int(epvar.numpy()),epochs):\n        rng=np.random.default_rng(seed*100000+epoch)\n        indices=rng.permutation(len(train))\n        losses=[];action_counts={"restore":0,"reject":0}\n        for k in indices:\n            row=train[int(k)]\n            x,y=load(row,size)\n            truth=y.numpy()[0,:,:,0]\n            p0,_=net.backbone(x,training=False)\n            pp=p0.numpy()[0,:,:,0]\n            if cfg["cdes"]=="natural":\n                mask,cands=candidates_from_model(pp,truth)\n            else:\n                mask,cands=candidates_from_cdes(pp,truth,rng,cfg["cdes"])\n            if not cands:continue\n            with tf.GradientTape() as tape:\n                pred,_,qr,qd,act=net.forward(x,cands,mask,training=False,evidence_mode=cfg["evidence"])\n                loss,components=training_loss(y,pred,qr,qd,cands,\n                    top_weight=.025,restore_weight=cfg["wr"],reject_weight=cfg["wd"])\n            train_vars=net.shared.trainable_variables+net.restore.trainable_variables+net.reject.trainable_variables\n            grads=tape.gradient(loss,train_vars)\n            valid=[(g,v) for g,v in zip(grads,train_vars) if g is not None]\n            if not valid or not all(bool(tf.reduce_all(tf.math.is_finite(g))) for g,v in valid):\n                raise RuntimeError("no valid finite head gradients")\n            optim.apply_gradients(valid)\n            losses.append(float(loss.numpy()))\n            for c in cands: action_counts[c["kind"]]+=1\n        observed=[];pair_rows=[]\n        for row in val:\n            x,y=load(row,size)\n            truth=y.numpy()[0,:,:,0]\n            p0,_=net.backbone(x,training=False)\n            pp=p0.numpy()[0,:,:,0]\n            mask,cs=candidates_for_inference(pp)\n            pred,_,qr,qd,_=net.forward(x,cs,mask,training=False,evidence_mode=cfg["evidence"])\n            real=pred.numpy()[0,:,:,0]\n            met=scores(truth,real)\n            base=scores(truth,pp)\n            met.update({"id":row["id"],"base_dice":base["dice"],"base_cldice":base["cldice"],\n                        "n_candidates":len(cs)})\n            observed.append(met)\n            # Ground truth is read only AFTER prediction for diagnostics, never proposal filtering.\n            for j,c in enumerate(cs):\n                gt_coverage=line_coverage(truth,c["a"],c["b"])\n                if .35<gt_coverage<.7:\n                    continue  # ambiguous edge has no binary evaluation label\n                truth_action=(gt_coverage>=.7) if c["kind"]=="restore" else (gt_coverage<=.35)\n                q=float(qr[j].numpy()) if c["kind"]=="restore" else float(qd[j].numpy())\n                pair_rows.append({"sample":row["id"],"kind":c["kind"],\n                                  "ground_truth_action_label":float(truth_action),\n                                  "acceptance_probability":q,"accepted":q>=.5})\n        if not observed:raise RuntimeError("validation has no records")\n        report={"method":"CETR-Net stage2 preliminary controlled experiment",\n                "status":"full_validation_only" if len(train)==508 and len(val)==90 else "validation_pilot_NOT_official_test",\n                "seed":seed,"variant":variant,"config":cfg,"epoch":epoch+1,\n                "train_count":len(train),"validation_count":len(val),\n                "train_loss_mean":float(np.mean(losses)) if losses else None,\n                "train_samples_with_candidates":len(losses),\n                "train_action_candidate_counts":action_counts,\n                "val_mean_dice":float(np.mean([r["dice"] for r in observed])),\n                "val_mean_cldice":float(np.mean([r["cldice"] for r in observed])),\n                "val_mean_iou":float(np.mean([r["iou"] for r in observed])),\n                "val_backbone_mean_dice":float(np.mean([r["base_dice"] for r in observed])),\n                "val_mean_delta_dice":float(np.mean([r["dice"]-r["base_dice"] for r in observed])),\n                "num_val_candidates":len(pair_rows),"validation_candidate_protocol":"prediction_only_before_metrics",\n                "seconds_elapsed":round(time.time()-now,1),\n                "paired_scores":observed,\n                "candidate_scores":pair_rows,\n                 "limitations":["Backbone is frozen during refiner stage and each seed must use its independently-trained matching backbone",\n                               "FIVES validation, NOT official held-out test",\n                               "Head-only optimization, not end-to-end training",\n                               "No independent MaskVSC baseline run",\n                               "Candidate 2D annotations are not 3D topology"]}\n        history.append(report)\n        atomic(out/"progress.json",report)\n        with (out/"epoch_history.jsonl").open("a",encoding="utf8") as f:\n            f.write(json.dumps({"epoch":epoch+1,"dice":report["val_mean_dice"],\n                       "cldice":report["val_mean_cldice"],"train_loss":report["train_loss_mean"]})+"\\n")\n        if report["val_mean_dice"]>best["best_dice"]:\n            best={"best_dice":report["val_mean_dice"],"best_epoch":epoch+1}\n            atomic(best_path,best)\n            np.savez_compressed(out/"best_head_weights.npz",\n              **{("shared_"+str(i)):w.numpy() for i,w in enumerate(net.shared.weights)},\n              **{("restore_"+str(i)):w.numpy() for i,w in enumerate(net.restore.weights)},\n              **{("reject_"+str(i)):w.numpy() for i,w in enumerate(net.reject.weights)})\n        epvar.assign(epoch+1)\n        manager.save(checkpoint_number=epoch+1)\n        print("EPOCH_STAGE2",seed,variant,epoch+1,round(report["val_mean_dice"],5),\n              round(report["val_mean_delta_dice"],5),"samples",len(losses),flush=True)\n    np.savez_compressed(out/"head_weights.npz",\n       **{("shared_"+str(i)):w.numpy() for i,w in enumerate(net.shared.weights)},\n       **{("restore_"+str(i)):w.numpy() for i,w in enumerate(net.restore.weights)},\n       **{("reject_"+str(i)):w.numpy() for i,w in enumerate(net.reject.weights)})\n    atomic(out/"completed.json",{"variant":variant,"seed":seed,"completed_at":time.strftime("%Y-%m-%d %H:%M:%S UTC",time.gmtime()),"final":history[-1]})\n    return history[-1]\ndef main():\n    p=argparse.ArgumentParser()\n    p.add_argument("--work",default="/kaggle/working/cetr_v2")\n    p.add_argument("--weights",required=True)\n    p.add_argument("--mode",choices=("preflight","pilot","fullstage"),default="pilot")\n    p.add_argument("--seed",type=int,default=42)\n    p.add_argument("--variant-list",type=str,default="",help="comma-separated predeclared variants")\n    p.add_argument("--epochs",type=int,default=8)\n    a=p.parse_args()\n    w=Path(a.work)\n    out=w/"CETR_V2_STAGE2_EXPERIMENTS"\n    out.mkdir(exist_ok=True,parents=True)\n    with (out/"RUN_STATUS.json").open("w") as f:json.dump({"mode":a.mode,"started":time.strftime("%Y-%m-%d %H:%M:%S UTC",time.gmtime())},f)\n    configs={"preflight":{"seeds":[42],"variants":["dual_full_asymmetric"],"train_n":4,"val_n":2,"epochs":1},\n             "pilot":{"seeds":[42,123,2025],"variants":list(VARIANTS),"train_n":24,"val_n":10,"epochs":1},\n             "fullstage":{"seeds":[42,123,2025],"variants":list(VARIANTS),"train_n":508,"val_n":90,"epochs":20}}\n    c=configs[a.mode]\n    if a.mode=="fullstage":\n        if a.seed not in (42,123,2025):raise ValueError("Unsupported FIVES V2 seed")\n        if a.epochs<1 or a.epochs>40:raise ValueError("Unsupported epoch setting")\n        c["seeds"]=[a.seed]\n        c["epochs"]=a.epochs\n        if a.variant_list:\n            variants=a.variant_list.split(",")\n            if len(set(variants))!=len(variants) or any(v not in VARIANTS for v in variants):\n                raise ValueError("Invalid or duplicate ablation variants")\n            c["variants"]=variants\n    print("FULLSTAGE_PROTOCOL",json.dumps(c),flush=True)\n    completed=[]\n    for seed in c["seeds"]:\n        for variant in c["variants"]:\n            print("START_STAGE2",seed,variant,flush=True)\n            try:\n                report=run_one(w,Path(a.weights),seed,variant,c["epochs"],c["train_n"],c["val_n"],512,out)\n                completed.append({"seed":seed,"variant":variant,\n                                  "val_mean_dice":report["val_mean_dice"],\n                                  "delta_dice":report["val_mean_delta_dice"],\n                                  "train_count":report["train_count"],"validation_count":report["validation_count"]})\n            except Exception as err:\n                error={"seed":seed,"variant":variant,"error":repr(err),"traceback":traceback.format_exc()}\n                atomic(out/"LAST_ERROR.json",error)\n                print("STAGE2_FAILED",json.dumps(error),flush=True)\n                raise\n            atomic(out/"SUMMARY.json",{"mode":a.mode,"completed":completed,\n                  "full_training_results":bool(a.mode=="fullstage"),\n                 "official_test_scored":False,\n                 "baseline_seed":int(a.seed) if a.mode=="fullstage" else "pretrained_seed42",\n                 "all_runs_passed":len(completed)==len(c["seeds"])*len(c["variants"])})\n    atomic(out/"RUN_STATUS.json",{"mode":a.mode,"status":"FINISHED","num_completed":len(completed)})\n    print("CETR_V2_STAGE2_COMPLETE",len(completed),flush=True)\nif __name__=="__main__":\n    main()\n',encoding='utf8')
import sys
sys.path.insert(0,str(WORK))
print('FULL_STAGE2_CODE_OK',len((WORK/'cetr_v2_suite.py').read_text()),flush=True)


In [ ]:

import os,json,hashlib
from pathlib import Path
seed=42
expected="3a6b1a472ec49779ec9e6464277a8ee2e299b3c5a1500af697c39a6b2b4c79d9"
found=[]
for here,dirs,files in os.walk("/kaggle/input",followlinks=True):
    if len(Path(here).relative_to("/kaggle/input").parts)>8:
        dirs.clear()
        continue
    if f"backbone_seed{seed}.weights.h5" in files:
        found.append(Path(here)/f"backbone_seed{seed}.weights.h5")
assert len(found)==1,"Migration checkpoint missing or ambiguous (expected unique Kaggle private model source)"
weights=found[0]
with weights.open("rb") as fp:
    got=hashlib.file_digest(fp,"sha256").hexdigest()
assert got==expected,"Migrated checkpoint differs from source SHA-256"
provenance_path=weights.parent/"SOURCE_MANIFEST.json"
assert provenance_path.is_file(),"Missing provenance manifest"
provenance=json.loads(provenance_path.read_text(encoding="utf8"))
assert provenance["seeds"][str(seed)]["sha256"]==expected
assert provenance["seeds"][str(seed)]["epochs_logged"]==45
assert provenance["official_test_scored"] is False
print("MIGRATED_BACKBONE_SHA256_OK",seed,got,flush=True)


In [ ]:

import json,sys,traceback,shutil,time
from cetr_v2_suite import main
seed=42
variants=['dual_geometry', 'dual_confidence', 'dual_full_asymmetric']
sys.argv=["cetr_v2_suite.py","--work",str(WORK),
          "--weights",str(weights),"--mode","fullstage",
          "--seed",str(seed),"--variant-list",",".join(variants),
          "--epochs",str(8)]
print("FULL_STAGE2_RUNNING",{"seed":seed,"group":'routing',"variants":variants,
  "epochs":8,"train":508,"validation":90},flush=True)
try:
    main()
finally:
    shutil.rmtree(WORK/"raw",ignore_errors=True)
print("FULL_STAGE2_KERNEL_COMPLETE",seed,'routing',flush=True)


In [ ]:

from pathlib import Path
import json
d=WORK/"CETR_V2_STAGE2_EXPERIMENTS"
s=json.loads((d/"SUMMARY.json").read_text())
print("FULLSTAGE_SUMMARY",s,flush=True)
assert s["full_training_results"] and s["all_runs_passed"]
assert not s["official_test_scored"]
